# 56 · Pathway approaches side by side: calls, specificity, overlap and the information each returns

Notebooks 31, 37 and 45 compared pathway screens for the species contrast under specimen
relabeling. This notebook completes that comparison with the approaches they did not run and puts
every approach in one table. Each approach answers a different question and returns a different
kind of information: yes/no, a sign, a segment, a curve, a location or a shape family.

| Part | Question |
|---|---|
| (a) | Thresholded **ORA on pseudobulk DE genes** passes our relabeling (118 species calls; 0 and 1 relabeled). Is that specificity, or the lower power of the relabeled design (1 against 2 residual df)? The test uses notebook 45's external same-species 2 + 2 splits (2 residual df, power-matched) and cross-species draws, plus equal probe counts |
| (b) | Self-contained **pathway activity scores** (mean-z module score and an AUCell-like rank score) along the coordinate, tested at specimen level (specimen × 8 bins, split plot) |
| (c) | **Curve modules**: k-means on the species-difference curves Δ(s) of the top 10% `T_spatial` genes, ORA per module; the same pipeline on each relabeling; leave-one-specimen-out stability |
| (d) | **Location enrichment**: are a pathway's members' divergence peaks more concentrated, or located differently, than matched random genes? |
| (e) | **tradeSeq-equivalent tests** inside the same nested Gaussian GAM: association per species, start-versus-end, early (S1) difference; gene-rank agreement with `T_spatial` |
| (f) | One taxonomy table and the overlap of every approach's species calls |

Rules were frozen in workstream B's phase-1 plan (section 8) before this notebook ran. Coordinate:
SCF13 (notebook 13's principal curve), the paper's coordinate. Cohort: 2 male control mice and 2
cortex sections from 1 male human donor; descriptive.

## 1 · Folders and pre-specified rules

| Rule | Value |
|---|---|
| Specificity (all parts) | notebook 31's rule: NCDR < 0.25 **and** each relabeling calls < 5% of tested pathways |
| (a) ORA | moderated-t pseudobulk (notebook 45's limma port) or saved DESeq2; DE gene = BH ≤ 0.05 and \|log2FC\| ≥ 1, each direction; hypergeometric against tested genes; BH over pathway × direction (× segment); a pathway is called if any test passes |
| (a) ORA is **specific** | 90th percentile of same-species null calls ≤ 5% of tested in all four pools **and** cross-species median NCDR < 0.25 **and** ≥ 80% of draws meet the rule; otherwise "passes our relabeling through power loss" |
| (a) probe exposure | ORA rerun on genes with equal probe counts in both panels; **probe-exposed** if ≥ 50% of the 118 species calls are not called |
| (b) scores | per structure: mean of species-balanced z-scored member expression; AUCell-like = mean over members of max(0, K − rank + 1)/K with K = 5% of genes; specimen × 8 equal bins of the common support (≥ 20 structures); split plot as notebook 37 (cubic in position; species × position against specimen × position); moderated F; BH ≤ 0.05 |
| (c) modules | top 10% genes by the partition's `T_spatial`; Δ(s) on 61 grid points, centred and scaled per gene; k-means (20 starts, seed 56); k ∈ 3–8 chosen on the species run by mean ARI over 20 resamples of 80% of genes (largest mean; ties → smaller k), then frozen; ORA per module against all tested genes, BH over module × pathway; **kept** if specific and median LOSO ARI ≥ 0.6 |
| (d) location | for the partition's top 10% genes, peak = grid position of max \|Δ(s)\|; pathways with ≥ 5 members in that set; concentration = MAD of member peaks (small is extreme), location = \|median member peak − median peak of the set\| (large is extreme); 9,999 random member sets of the same size drawn within notebook 37's strata from the same top-10% genes; BH per statistic |
| (e) | gene-rank Spearman with species `T_spatial` and top-10% overlap; no pathway calls |

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
coordinate_file = results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv'
nb31 = results_root / 'pt_pathway_method_selection'
nb37 = results_root / 'pt_pathway_final' / 'scfates'
nb45 = results_root / 'pt_revision_method'
nb12 = results_root / 'pt_pathway_remodeling_scfates'
probe_file = results_root / 'pt_literature_deep' / 'probe_counts.csv'
primary_file = results_root / 'pt_primary_pathways' / 'tables' / 'primary_pathway_list.csv'
step45_file = results_root / 'pt_primary_pathways' / 'tables' / 'cohort_step_screen_per_pathway.csv'
output = results_root / 'pt_pathway_continuous' / '56_taxonomy'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)
for path in (coordinate_file, nb31 / 'conventional_deseq2_all_partitions.csv', nb31 / 'strategy_hits_all_partitions.csv',
             nb31 / 'conventional_signed_gsea_all_partitions.csv', nb37 / 'tables' / 'matched_and_joint_tests_all_partitions.csv',
             nb37 / 'tables' / 'table_S1_strategy_specificity.csv', nb37 / 'tables' / 'table_S2_pathways.csv',
             nb45 / 'tables' / 'table_cross_species_draws_summary.csv', nb12 / 'conventional_pt_ora.csv',
             probe_file, primary_file, step45_file):
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '56.taxonomy.1'  # Bump when a cached calculation changes.
ALPHA, NULL_RATE, NCDR_MAX = .05, .05, .25
LFC = 1.0
POOL_SEED, N_SETS, N_CROSS = 45, 100, 100          # notebook 45's draws, reproduced exactly
N_BINS, MIN_CELL = 8, 20                           # notebook 37's split plot
AUCELL_TOP = .05
TOP_SHARE, K_RANGE, N_RESAMPLE, KM_SEED, KM_STARTS = .10, range(3, 9), 20, 56, 20
MIN_MEMBERS_LOCATION, N_NULL_LOCATION, LOCATION_SEED = 5, 9999, 56
print('Results folder:', output)

## 2 · Our structures, genes and pathways on SCF13

In [ ]:
import glob
from itertools import combinations

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from scipy.stats import hypergeom, spearmanr
from statsmodels.stats.multitest import multipletests

from pseudospace import continuous_descriptors as descriptors_module
from pseudospace import coordinate_inputs as inputs_module
from pseudospace.continuous_descriptors import group_difference_curves
from pseudospace.coordinate_inputs import pt_inputs, read_coordinate
from pseudospace.pathway_calibration import balanced_partitions, contrast_designs, nested_partial_f, pathway_scores
from pseudospace.split_plot import log_cpm, moderated_f, voom_weights
from pseudospace.stage_cache import cached_frame, cached_payload, digest
from pseudospace.stats_gam import gam_internal_knots, make_gam_design

pd.set_option('display.width', 220, 'display.max_columns', 30, 'display.max_colwidth', 70)
inputs = pt_inputs(results_root, data_root, read_coordinate(coordinate_file))
position, human, specimen, segment = (inputs[k] for k in ('position', 'human', 'specimen', 'segment'))
genes, gene_sets, strata = inputs['genes'], inputs['gene_sets'], inputs['strata']
gene_index = pd.Index(genes)
if len(genes) != 11071 or len(gene_sets) != 1513:  # Guard: notebook 37's universe on SCF13.
    raise ValueError(f'Expected 11,071 genes and 1,513 pathways, found {len(genes)} and {len(gene_sets)}.')
pathways = list(gene_sets)
partitions = balanced_partitions(specimen, np.where(human, 'human', 'mouse'))
swaps = [p for p in partitions if p != 'species']
table37 = pd.read_csv(nb37 / 'tables' / 'table_S2_pathways.csv').set_index('pathway_id')
R36 = set(table37.index[table37.robust.astype(bool)])
P22 = set(pd.read_csv(primary_file).pathway_id)
cache = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(descriptors_module.__file__), Path(inputs_module.__file__)])
input_key = {'Y': digest(inputs['Y']), 'position': position, 'specimen': specimen, 'genes': genes}


def membership(sets, universe):
    """Pathways x genes 0/1 matrix over ``universe``."""
    index = pd.Index(universe)
    rows, cols = [], []
    for i, members in enumerate(sets.values()):
        idx = index.get_indexer(members)
        idx = idx[idx >= 0]
        rows.extend([i] * len(idx)), cols.extend(idx)
    return sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(sets), len(index)))


def ora_pvalues(selected, member_matrix):
    """One-sided hypergeometric p per pathway for a boolean gene selection."""
    selected = np.asarray(selected, bool)
    n_sel, n_all = int(selected.sum()), len(selected)
    if n_sel == 0:
        return np.ones(member_matrix.shape[0])
    k = np.asarray(member_matrix @ selected.astype(float)).ravel()
    size = np.asarray(member_matrix.sum(axis=1)).ravel()
    return hypergeom.sf(k - 1, n_all, size, n_sel)


def family_calls(pvalue_columns):
    """BH over every column of a pathway x test p matrix; called if any test passes."""
    p = np.column_stack(pvalue_columns)
    q = multipletests(p.ravel(), method='fdr_bh')[1].reshape(p.shape)
    return (q <= ALPHA).any(axis=1)


def rule(species_calls, relabel_calls, tested):
    ncdr = np.mean(relabel_calls) / species_calls if species_calls else np.nan
    return ncdr, bool(species_calls and ncdr < NCDR_MAX and max(relabel_calls) < NULL_RATE * tested)


print(f'{len(position):,} structures · {len(genes):,} genes · {len(gene_sets):,} pathways')

## 3 · (a) ORA on pseudobulk DE genes

### Our cohort, from notebook 31's saved DESeq2 (species split and both relabelings)

The relabeled DESeq2 design is `~species + group` (1 residual df), the species design `~species`
(2 residual df). The equal-probe rerun restricts genes to equal probe counts in both panels and
recomputes BH within them.

In [ ]:
deseq = pd.read_csv(nb31 / 'conventional_deseq2_all_partitions.csv')
probes = pd.read_csv(probe_file).drop_duplicates('mouse_symbol').set_index('mouse_symbol')
equal_probe = set(probes.index[probes.mouse_probes.notna() & probes.mouse_probes.eq(probes.human_probes)])


def restrict(sets, keep):
    keep = set(keep)
    out = {p: [g for g in v if g in keep] for p, v in sets.items()}
    return {p: v for p, v in out.items() if 10 <= len(v) <= min(300, len(keep) - 1)}


def deseq_ora(frame, sets, keep=None):
    calls = {}
    for family, scopes in (('S1/S2/S3', ['PT-S1', 'PT-S2', 'PT-S3']), ('whole PT', ['whole_PT'])):
        cols = []
        for scope in scopes:
            rows = frame[frame.scope.eq(scope)].dropna(subset=['stat']).set_index('gene')
            if keep is not None:
                rows = rows[rows.index.isin(sorted(keep))].copy()
                rows['padj'] = multipletests(rows.pvalue.fillna(1.), method='fdr_bh')[1]
            universe = list(rows.index)
            matrix = membership(sets, universe)
            for direction in (1, -1):
                selected = rows.padj.le(ALPHA).to_numpy() & (direction * rows.log2FoldChange).ge(LFC).to_numpy()
                cols.append(ora_pvalues(selected, matrix))
        calls[family] = pd.Series(family_calls(cols), index=list(sets))
    return calls


ora_rows, ora_sets = [], {}
for label, frame in deseq.groupby('partition'):
    for variant, sets, keep in (('all genes', gene_sets, None), ('equal probe counts', restrict(gene_sets, equal_probe), equal_probe)):
        for family, hits in deseq_ora(frame, sets, keep).items():
            ora_sets[(variant, family, label)] = set(hits.index[hits])
            ora_rows.append({'variant': variant, 'family': family, 'partition': label, 'tested': len(sets),
                             'calls': int(hits.sum())})
ORA_OURS = pd.DataFrame(ora_rows)
display(ORA_OURS.pivot_table(index=['variant', 'family'], columns='partition', values='calls'))
species_ora = ora_sets[('all genes', 'S1/S2/S3', 'species')]
kept = species_ora & ora_sets[('equal probe counts', 'S1/S2/S3', 'species')]
probe_untestable = species_ora - set(restrict(gene_sets, equal_probe))
probe_lost_share = 1 - len(kept) / max(len(species_ora), 1)
print(f'S1/S2/S3 ORA species calls {len(species_ora)}; still called on equal-probe genes {len(kept)} '
      f'({len(probe_untestable)} untestable there); share lost {probe_lost_share:.2f} → '
      f'{"probe-exposed" if probe_lost_share >= .5 else "not probe-exposed"}')

### External same-species and cross-species 2 + 2 draws (notebook 45's pools and draws)

Notebook 45 cached one payload per pool (donor × segment count sums, tested genes, pathways). Its
draws are reproduced exactly from its seeds. A same-species split has design `~group` (2 residual
df), the same as our species design: this is the power-matched null for ORA.

In [ ]:
nb45_cache = nb45 / 'stage_cache'
POOLS = {'N1 Census mouse, male': 'pool_census_mouse_male', 'N2 Census mouse, female': 'pool_census_mouse_female',
         'N3 Lake/KPMP human cortex': 'pool_lake_human', 'N4 Census human cortex': 'pool_census_human',
         'N5 Lake human + Census male mouse': 'pool_cross_species', 'N7 our cohort': 'pool_our_cohort'}


def newest_payload(stage):
    files = glob.glob(str(nb45_cache / f'{stage}__*.npz'))
    if not files:
        raise FileNotFoundError(f'Notebook 45 cache {stage} not found.')
    files.sort(key=lambda p: json.loads(Path(p + '.key.json').read_text())['written_utc'])
    with np.load(files[-1]) as stored:
        return {k: stored[k] for k in stored.files}


# Notebook 45 built its pathway sets from notebook 12's universe (members = tested = universe genes).
from pseudospace.pathways import build_pathway_membership

universe12 = pd.read_csv(nb12 / 'gene_statistics.csv', index_col=0).index.tolist()
coverage45 = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json').read_text()), universe12,
    ortholog_map=pd.read_csv(results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv'), library_name=lib,
    tested=universe12, min_genes=1) for lib in ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')], ignore_index=True)
coverage45['pathway_id'] = coverage45.library + '::' + coverage45.pathway
coverage45 = coverage45[coverage45.n_tested.between(10, min(300, len(universe12) - 1))]
gene_sets_45 = {row.pathway_id: list(row.genes_present) for row in coverage45.itertuples()}
pools = {}
for name, stage in POOLS.items():
    pool = newest_payload(stage)
    pool['sets'] = restrict(gene_sets_45, set(pool['genes'].astype(str)))
    if list(pool['sets']) != list(pool['pathways'].astype(str)):  # Guard: notebook 45's pathway sets.
        raise ValueError(f'{name}: pathways differ from notebook 45.')
    pool['matrix'] = membership(pool['sets'], list(pool['genes'].astype(str)))
    pools[name] = pool
display(pd.DataFrame({n: {'donors': len(p['donors']), 'tested genes': len(p['genes']), 'pathways': len(p['sets'])}
                      for n, p in pools.items()}).T)


def pseudobulk_ora(pool, donors, group, nuisance=None):
    """ORA calls (whole PT, any segment) and DE gene counts for one 2 + 2 partition."""
    index = pd.Index(pool['donors'].astype(str)).get_indexer(donors)
    columns = [np.ones(len(donors)), [group[d] for d in donors]]
    if nuisance is not None:
        columns.append([nuisance[d] for d in donors])
    design = np.column_stack(columns).astype(float)
    tests, de_counts = {}, {}
    scopes = [('whole', pool['counts'][index].sum(axis=1), pool['library'][index].sum(axis=1))]
    scopes += [(seg, pool['counts'][index, k], pool['library'][index, k]) for k, seg in enumerate(pool['segments'].astype(str))]
    for scope, counts, library in scopes:
        y = log_cpm(counts, library)
        fit = moderated_f(y, voom_weights(y, library, design), design, [1])
        q = multipletests(fit['p'], method='fdr_bh')[1]
        beta = fit['beta'][:, 1]
        cols = []
        for direction in (1, -1):
            selected = (q <= ALPHA) & (direction * beta >= LFC)
            de_counts[scope] = de_counts.get(scope, 0) + int(selected.sum())
            cols.append(ora_pvalues(selected, pool['matrix']))
        tests[scope] = cols
    whole = family_calls(tests['whole'])
    segments = family_calls([c for s in pool['segments'].astype(str) for c in tests[s]])
    return {'ORA whole PT': whole, 'ORA S1/S2/S3': segments, 'DE genes (median over scopes)': float(np.median(list(de_counts.values())))}


def same_species_partitions(donors, rng):
    all_sets = list(combinations(sorted(donors), 4))
    chosen = all_sets if len(all_sets) <= N_SETS else [all_sets[i] for i in sorted(rng.choice(len(all_sets), N_SETS, replace=False))]
    parts = []
    for i, (a, b, c, d) in enumerate(chosen):
        for j, (first, second) in enumerate((((a, b), (c, d)), ((a, c), (b, d)), ((a, d), (b, c)))):
            parts.append({'label': f'set{i:03d}_split{j}', 'donors': [a, b, c, d],
                          'group': {x: float(x in second) for x in (a, b, c, d)}})
    return parts


import multiprocessing
from concurrent.futures import ProcessPoolExecutor

WORKERS = int(os.environ.get('PSEUDOSPACE_WORKERS', 16))


def _worker_init():
    from threadpoolctl import threadpool_limits
    threadpool_limits(1)


def _partition_job(job):
    pool_name, part = job
    return pseudobulk_ora(pools[pool_name], part['donors'], part['group'], part.get('nuisance'))


def run_partitions(pool_name, parts, stage):
    def compute():
        with ProcessPoolExecutor(max_workers=WORKERS, mp_context=multiprocessing.get_context('fork'),
                                 initializer=_worker_init) as executor:
            results = list(executor.map(_partition_job, [(pool_name, p) for p in parts], chunksize=4))
        rows = []
        for part, result in zip(parts, results):
            for strategy in ('ORA whole PT', 'ORA S1/S2/S3'):
                rows.append({'pool': pool_name, 'label': part['label'], 'strategy': strategy,
                             'tested': len(pools[pool_name]['sets']), 'calls': int(result[strategy].sum()),
                             'called': '|'.join(map(str, np.flatnonzero(result[strategy]))),
                             'DE genes (median over scopes)': result['DE genes (median over scopes)'],
                             **{k: v for k, v in part.items() if k not in ('donors', 'group', 'nuisance', 'label')}})
        return pd.DataFrame(rows)
    spec = [{k: (sorted(v.items()) if isinstance(v, dict) else v) for k, v in p.items()} for p in parts]
    return cached_frame(stage, compute, root=cache, params={'alpha': ALPHA, 'lfc': LFC},
                        inputs={'pool': digest(pools[pool_name]['counts']), 'parts': spec}, code=code)


rng = np.random.default_rng(POOL_SEED)
null_frames = []
for name in ['N1 Census mouse, male', 'N2 Census mouse, female', 'N3 Lake/KPMP human cortex', 'N4 Census human cortex']:
    parts = same_species_partitions(list(pools[name]['donors'].astype(str)), rng)
    null_frames.append(run_partitions(name, parts, 'ora_null_' + name.split()[0]))
ORA_NULL = pd.concat(null_frames, ignore_index=True)
ORA_NULL.drop(columns='called').to_csv(tables / 'a_ora_same_species_null_calls.csv', index=False)
null_summary = ORA_NULL.groupby(['pool', 'strategy']).apply(lambda f: pd.Series({
    'splits': len(f), 'tested': int(f.tested.iloc[0]), 'median calls': f.calls.median(), 'p90 calls': f.calls.quantile(.9),
    'share of splits ≥ 5% of tested': (f.calls / f.tested).ge(NULL_RATE).mean(),
    'median DE genes': f['DE genes (median over scopes)'].median()}), include_groups=False)
display(null_summary)

lake_donors = sorted(pools['N3 Lake/KPMP human cortex']['donors'].astype(str))
mouse_male = sorted(pools['N1 Census mouse, male']['donors'].astype(str))
species_of = {**{d: 1. for d in lake_donors}, **{d: 0. for d in mouse_male}}
all_draws = [(h, m) for h in combinations(lake_donors, 2) for m in combinations(mouse_male, 2)]
rng_cross = np.random.default_rng(POOL_SEED + 1)
draws = [all_draws[i] for i in sorted(rng_cross.choice(len(all_draws), N_CROSS, replace=False))]
cross_parts = []
for i, ((h1, h2), (m1, m2)) in enumerate(draws):
    four = [h1, h2, m1, m2]
    cross_parts.append({'label': f'draw{i:03d}_species', 'donors': four, 'group': {d: species_of[d] for d in four},
                        'draw': i, 'partition': 'species'})
    for j, partner in enumerate((m1, m2)):
        cross_parts.append({'label': f'draw{i:03d}_relabel{j}', 'donors': four,
                            'group': {d: float(d in {h1, partner}) for d in four},
                            'nuisance': {d: species_of[d] for d in four}, 'draw': i, 'partition': f'relabel{j}'})
ORA_CROSS = run_partitions('N5 Lake human + Census male mouse', cross_parts, 'ora_cross_species')
ORA_CROSS.drop(columns='called').to_csv(tables / 'a_ora_cross_species_draws.csv', index=False)


def draw_metrics(frame):
    species = frame[frame.partition.eq('species')].iloc[0]
    relabel = frame[frame.partition.ne('species')]
    ncdr, meets = rule(species.calls, list(relabel.calls), species.tested)
    return pd.Series({'species calls': species.calls, 'relabel mean': relabel.calls.mean(), 'ncdr': ncdr, 'meets rule': meets})


cross_summary = ORA_CROSS.groupby(['strategy', 'draw']).apply(draw_metrics, include_groups=False).groupby('strategy').agg(
    species_calls_median=('species calls', 'median'), relabel_calls_median=('relabel mean', 'median'),
    ncdr_median=('ncdr', 'median'), share_meeting_rule=('meets rule', 'mean'))
display(cross_summary)

anchor_parts = [{'label': 'species', 'donors': ['Ctrl1A2', 'Ctrl1A4', 'HUK1_COR1', 'HUK1_MED1'],
                 'group': {'Ctrl1A2': 0., 'Ctrl1A4': 0., 'HUK1_COR1': 1., 'HUK1_MED1': 1.}}]
for partner in ('Ctrl1A2', 'Ctrl1A4'):
    group = {d: float(d in {'HUK1_COR1', partner}) for d in ['Ctrl1A2', 'Ctrl1A4', 'HUK1_COR1', 'HUK1_MED1']}
    anchor_parts.append({'label': f'relabel HUK1_COR1+{partner}', 'donors': list(group), 'group': group,
                         'nuisance': {'Ctrl1A2': 0., 'Ctrl1A4': 0., 'HUK1_COR1': 1., 'HUK1_MED1': 1.}})
    anchor_parts.append({'label': f'relabel HUK1_COR1+{partner}, no species covariate (2 df)', 'donors': list(group),
                         'group': group})
ORA_ANCHOR = run_partitions('N7 our cohort', anchor_parts, 'ora_our_cohort')
display(ORA_ANCHOR.drop(columns='called').pivot_table(index='label', columns='strategy', values=['calls', 'DE genes (median over scopes)']))

decision_rows = []
for strategy in ('ORA whole PT', 'ORA S1/S2/S3'):
    p90_ok = bool(null_summary.xs(strategy, level='strategy').pipe(lambda d: (d['p90 calls'] <= NULL_RATE * d['tested']).all()))
    cross = cross_summary.loc[strategy]
    specific = p90_ok and cross.ncdr_median < NCDR_MAX and cross.share_meeting_rule >= .8
    decision_rows.append({'strategy': strategy, 'null p90 ≤ 5% in all four pools': p90_ok,
                          'cross-species median NCDR': cross.ncdr_median, 'share of draws meeting rule': cross.share_meeting_rule,
                          'decision': 'specific' if specific else 'passes our relabeling through power loss'})
ORA_DECISION = pd.DataFrame(decision_rows)
ORA_DECISION['probe exposure (S1/S2/S3, our cohort)'] = f'{probe_lost_share:.2f} of species calls lost → ' + (
    'probe-exposed' if probe_lost_share >= .5 else 'not probe-exposed')
ORA_DECISION.to_csv(tables / 'a_ora_decision.csv', index=False)
ORA_OURS.to_csv(tables / 'a_ora_our_cohort_deseq2.csv', index=False)
display(ORA_DECISION)

## 4 · (b) Pathway activity scores along the coordinate, tested at specimen level

Notebook 31's structure-level score models called nearly everything, relabeled or not, because
12,000 structures are not replicates. Here each score is averaged in specimen × position cells and
tested in the split plot, so the error term is specimen × position variation (as in notebook 37).

In [ ]:
Y = inputs['Y'].toarray()
member = membership(gene_sets, genes)
member_size = np.asarray(member.sum(axis=1)).ravel()
weights = np.zeros(len(position))
for sp in (True, False):
    names = np.unique(specimen[human == sp])
    for n in names:
        rows = specimen == n
        weights[rows] = len(position) / (2 * len(names) * rows.sum())


def structure_scores():
    meanz = pathway_scores(Y, gene_sets, genes, weights).to_numpy(float)
    k = int(round(AUCELL_TOP * len(genes)))
    ranks = np.empty_like(Y, dtype=np.float32)
    order = np.argsort(-Y, axis=1, kind='stable')
    np.put_along_axis(ranks, order, np.arange(1, len(genes) + 1, dtype=np.float32)[None, :], axis=1)
    credit = np.clip(k - ranks + 1, 0, None) / k
    aucell = (credit @ member.T.toarray().astype(np.float32)).astype(float) / member_size[None, :]
    return {'mean_z': meanz, 'aucell': aucell}


scores = cached_payload('structure_scores', structure_scores, root=cache, params={'top': AUCELL_TOP},
                        inputs={**input_key, 'sets': gene_sets}, code=code)
lo, hi = inputs['grid'][0], inputs['grid'][-1]
bins = np.clip(((position - lo) / (hi - lo) * N_BINS).astype(int), 0, N_BINS - 1)
cells = pd.DataFrame({'specimen': specimen, 'bin': bins})
cell_keys = cells.groupby(['specimen', 'bin']).size()
cell_keys = cell_keys[cell_keys >= MIN_CELL]
names = ['Ctrl1A2', 'Ctrl1A4', 'HUK1_COR1', 'HUK1_MED1']


def split_plot_scores(matrix):
    """Moderated F of group x position against specimen x position on specimen x bin means."""
    frame = pd.DataFrame(matrix)
    means = frame.groupby([specimen, bins]).mean().loc[cell_keys.index]
    cell_spec = means.index.get_level_values(0).to_numpy()
    centre = (means.index.get_level_values(1).to_numpy() + .5) / N_BINS
    basis = make_gam_design(centre, np.array([]))[:, 1:]
    spec_cols = (cell_spec[:, None] == np.array(names)[None, :]).astype(float)
    cell_human = np.isin(cell_spec, ['HUK1_COR1', 'HUK1_MED1']).astype(float)
    out = {}
    for label in partitions:
        if label == 'species':
            design = np.column_stack([spec_cols, basis, cell_human[:, None] * basis])
        else:
            group = np.isin(cell_spec, partitions[label]).astype(float)
            design = np.column_stack([spec_cols, basis, cell_human[:, None] * basis, group[:, None] * basis])
        test = list(range(design.shape[1] - basis.shape[1], design.shape[1]))
        fit = moderated_f(means.to_numpy(float), np.ones(means.shape), design, test)
        out[label] = multipletests(fit['p'], method='fdr_bh')[1] <= ALPHA
    return out


score_rows, score_sets = [], {}
for score_name in ('mean_z', 'aucell'):
    calls = split_plot_scores(scores[score_name])
    ncdr, meets = rule(int(calls['species'].sum()), [int(calls[s].sum()) for s in swaps], len(pathways))
    score_sets[score_name] = {k: set(np.array(pathways)[v]) for k, v in calls.items()}
    score_rows.append({'score': score_name, 'species calls': int(calls['species'].sum()),
                       **{f'relabel {s.split("+")[-1]}': int(calls[s].sum()) for s in swaps}, 'NCDR': ncdr,
                       'meets rule': meets, '∩ 22': len(score_sets[score_name]['species'] & P22),
                       '∩ 36': len(score_sets[score_name]['species'] & R36)})
SCORES = pd.DataFrame(score_rows)
SCORES.to_csv(tables / 'b_pathway_score_split_plot.csv', index=False)
print(f'{len(cell_keys)} specimen × bin cells')
display(SCORES)

## 5 · (c) Curve modules of the species-difference curves, with ORA

Δ(s) for each partition comes from notebook 31's full design (relabelings carry the species shape
as a nuisance). Genes are the partition's top 10% by `T_spatial`.

In [ ]:
knots = gam_internal_knots(position, basis_df=6)
grid = inputs['grid']


def partition_curves(label, keep=None):
    rows = np.ones(len(position), bool) if keep is None else keep
    if label == 'species':
        group, nuisance = human.astype(float), None
    else:
        group, nuisance = np.isin(specimen, partitions[label]).astype(float), human.astype(float)
    delta = group_difference_curves(Y[rows], position[rows], group[rows], specimen[rows], knots, grid,
                                    nuisance_shape=None if nuisance is None else nuisance[rows])
    designs, w = contrast_designs(position[rows], group[rows], specimen[rows], knots,
                                  nuisance_shape=None if nuisance is None else nuisance[rows])
    t_spatial = nested_partial_f(Y[rows], designs, w, {'T_spatial': ('level', 'full')})['T_spatial']
    return {'delta': delta, 'T_spatial': t_spatial}


curves = {label: cached_payload('curves_' + label.replace(':', '_').replace('+', '_'),
                                lambda label=label: partition_curves(label), root=cache, params={'label': label},
                                inputs=input_key, code=code) for label in partitions}
reference37 = pd.read_csv(nb37 / 'tables' / 'gene_level_statistics.csv', index_col=0).T_spatial
np.testing.assert_allclose(curves['species']['T_spatial'], reference37.reindex(genes), rtol=1e-6)  # Guard


def standardized_top(curve):
    top = np.argsort(-curve['T_spatial'])[:int(round(TOP_SHARE * len(genes)))]
    d = curve['delta'][top]
    d = (d - d.mean(axis=1, keepdims=True)) / np.maximum(d.std(axis=1, keepdims=True), 1e-12)
    return np.sort(top), d[np.argsort(top)]


from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score


def kmeans(data, k, seed=KM_SEED):
    return KMeans(n_clusters=k, n_init=KM_STARTS, random_state=seed).fit(data)


top_species, data_species = standardized_top(curves['species'])
stability = {}
rng_k = np.random.default_rng(KM_SEED)
for k in K_RANGE:
    full = kmeans(data_species, k).labels_
    aris = []
    for r in range(N_RESAMPLE):
        pick = np.sort(rng_k.choice(len(data_species), int(.8 * len(data_species)), replace=False))
        aris.append(adjusted_rand_score(full[pick], kmeans(data_species[pick], k, seed=KM_SEED + r + 1).labels_))
    stability[k] = float(np.mean(aris))
K = max(stability, key=lambda k: (round(stability[k], 6), -k))
print('Mean resample ARI by k:', {k: round(v, 3) for k, v in stability.items()}, '→ k =', K)


def module_ora(top, labels):
    universe = np.zeros(len(genes), bool)
    cols = []
    for m in range(K):
        selected = universe.copy()
        selected[top[labels == m]] = True
        cols.append(ora_pvalues(selected, member))
    return family_calls(cols), np.column_stack(cols)


module_sets, module_rows = {}, []
species_model = kmeans(data_species, K)
for label in partitions:
    top, data = (top_species, data_species) if label == 'species' else standardized_top(curves[label])
    labels = species_model.labels_ if label == 'species' else kmeans(data, K).labels_
    hits, _ = module_ora(top, labels)
    module_sets[label] = set(np.array(pathways)[hits])
ncdr, meets = rule(len(module_sets['species']), [len(module_sets[s]) for s in swaps], len(pathways))

loso_ari = {}
for name in names:
    keep = specimen != name
    loso = cached_payload('curves_loso_' + name, lambda keep=keep: partition_curves('species', keep), root=cache,
                          params={'omit': name}, inputs=input_key, code=code)
    d = loso['delta'][top_species]
    d = (d - d.mean(axis=1, keepdims=True)) / np.maximum(d.std(axis=1, keepdims=True), 1e-12)
    loso_ari[name] = adjusted_rand_score(species_model.labels_, kmeans(d, K).labels_)
MODULES = pd.DataFrame([{'k': K, 'species calls': len(module_sets['species']),
                         **{f'relabel {s.split("+")[-1]}': len(module_sets[s]) for s in swaps}, 'NCDR': ncdr,
                         'meets rule': meets, 'median LOSO ARI': float(np.median(list(loso_ari.values()))),
                         **{f'LOSO ARI omit {n}': v for n, v in loso_ari.items()},
                         'kept': bool(meets and np.median(list(loso_ari.values())) >= .6),
                         '∩ 22': len(module_sets['species'] & P22), '∩ 36': len(module_sets['species'] & R36)}])
MODULES.to_csv(tables / 'c_curve_modules.csv', index=False)
centroids = pd.DataFrame(species_model.cluster_centers_, columns=np.round(grid, 3))
centroids['genes'] = np.bincount(species_model.labels_, minlength=K)
centroids['peak position'] = grid[np.argmax(np.abs(species_model.cluster_centers_), axis=1)]
centroids.to_csv(tables / 'c_module_centroids.csv')
pd.DataFrame({'gene': np.array(genes)[top_species], 'module': species_model.labels_}).to_csv(tables / 'c_module_genes.csv', index=False)
display(MODULES)
display(centroids[['genes', 'peak position']])

## 6 · (d) Location enrichment of divergence peaks

Each gene's peak is the grid position of its largest \|Δ(s)\|. A pathway can be extreme by having
its members' peaks close together (concentration) or by peaking in a different place from the
other strongly divergent genes (location).

In [ ]:


def location_tests(label):
    curve = curves[label]
    top = np.argsort(-curve['T_spatial'])[:int(round(TOP_SHARE * len(genes)))]
    peaks = pd.Series(grid[np.argmax(np.abs(curve['delta'][top]), axis=1)], index=np.array(genes)[top])
    pool_strata = pd.Series(np.asarray(strata), index=genes).loc[peaks.index].to_numpy()
    centre = peaks.median()
    rng = np.random.default_rng(LOCATION_SEED)
    pools_by_stratum = {s: np.flatnonzero(pool_strata == s) for s in np.unique(pool_strata)}
    peak_values = peaks.to_numpy()
    drawn = {}  # (stratum, count) -> draws x count peak values, shared by every pathway as in notebook 12

    def stratum_draws(s, count):
        if (s, count) not in drawn:
            pool_s = pools_by_stratum[s]
            order = np.argsort(rng.random((N_NULL_LOCATION, len(pool_s))), axis=1)[:, :count]
            drawn[(s, count)] = peak_values[pool_s[order]]
        return drawn[(s, count)]

    rows = []
    for p, members in gene_sets.items():
        idx = peaks.index.get_indexer([g for g in members if g in peaks.index])
        if len(idx) < MIN_MEMBERS_LOCATION:
            continue
        values = peak_values[idx]
        mad = np.median(np.abs(values - np.median(values)))
        shift = abs(np.median(values) - centre)
        draws = np.column_stack([stratum_draws(s, c) for s, c in zip(*np.unique(pool_strata[idx], return_counts=True))])
        med = np.median(draws, axis=1)
        null_mad = np.median(np.abs(draws - med[:, None]), axis=1)
        null_shift = np.abs(med - centre)
        rows.append({'pathway_id': p, 'members in top set': len(idx), 'median peak': float(np.median(values)),
                     'MAD': mad, 'p concentration': (np.sum(null_mad <= mad) + 1) / (N_NULL_LOCATION + 1),
                     'p location': (np.sum(null_shift >= shift) + 1) / (N_NULL_LOCATION + 1)})
    frame = pd.DataFrame(rows)
    for stat in ('concentration', 'location'):
        frame[f'q {stat}'] = multipletests(frame[f'p {stat}'], method='fdr_bh')[1]
    return frame


location = {label: cached_frame('location_' + label.replace(':', '_').replace('+', '_'),
                                lambda label=label: location_tests(label), root=cache,
                                params={'n': N_NULL_LOCATION, 'seed': LOCATION_SEED, 'min': MIN_MEMBERS_LOCATION},
                                inputs=input_key, code=code) for label in partitions}
location_rows, location_sets = [], {}
for stat in ('concentration', 'location'):
    sets_ = {label: set(frame.pathway_id[frame[f'q {stat}'] <= ALPHA]) for label, frame in location.items()}
    location_sets[stat] = sets_
    ncdr, meets = rule(len(sets_['species']), [len(sets_[s]) for s in swaps], len(location['species']))
    location_rows.append({'statistic': stat, 'tested (species)': len(location['species']),
                          'species calls': len(sets_['species']),
                          **{f'relabel {s.split("+")[-1]}': len(sets_[s]) for s in swaps}, 'NCDR': ncdr, 'meets rule': meets,
                          '∩ 22': len(sets_['species'] & P22), '∩ 36': len(sets_['species'] & R36)})
LOCATION = pd.DataFrame(location_rows)
LOCATION.to_csv(tables / 'd_location_enrichment.csv', index=False)
location['species'].to_csv(tables / 'd_location_species_per_pathway.csv', index=False)
display(LOCATION)

## 7 · (e) tradeSeq-equivalent tests in the same nested GAM

- **Association per species** (tradeSeq `associationTest`): spline along position against a
  specimen offset only, within each species.
- **Start versus end** (`startVsEndTest` applied to the species difference): \|Δ(end) − Δ(start)\|.
- **Early difference** (`earlyDETest`, restricted to S1-labelled structures): species × spline
  beyond a species offset within S1.

In [ ]:


def association(rows):
    s, spec = position[rows], specimen[rows]
    names_ = np.unique(spec)
    nuisance = np.column_stack([(spec == n).astype(float) - (spec == names_[-1]).astype(float) for n in names_[:-1]])
    base = make_gam_design(s, knots)
    designs = {'offset': np.column_stack([np.ones(rows.sum()), nuisance]), 'spline': np.column_stack([base, nuisance])}
    return nested_partial_f(Y[rows], designs, np.ones(rows.sum()), {'F': ('offset', 'spline')})['F']


def early_difference():
    rows = segment == 'PT-S1'
    designs, w = contrast_designs(position[rows], human[rows].astype(float), specimen[rows],
                                  gam_internal_knots(position[rows], basis_df=4))
    return nested_partial_f(Y[rows], designs, w, {'F': ('level', 'full')})['F']


tradeseq = pd.DataFrame(cached_payload('tradeseq_equivalents', lambda: {
    'association mouse': association(~human), 'association human': association(human),
    'start vs end': np.abs(curves['species']['delta'][:, -1] - curves['species']['delta'][:, 0]),
    'early (S1) difference': early_difference()}, root=cache, inputs=input_key, code=code), index=genes)
t_spatial = pd.Series(curves['species']['T_spatial'], index=genes)
top_t = set(t_spatial.nlargest(int(round(TOP_SHARE * len(genes)))).index)
TRADESEQ = pd.DataFrame([{'statistic': c, 'gene Spearman with T_spatial': spearmanr(tradeseq[c], t_spatial).statistic,
                          'top-10% overlap with T_spatial': len(set(tradeseq[c].nlargest(len(top_t)).index) & top_t) / len(top_t)}
                         for c in tradeseq])
TRADESEQ.to_csv(tables / 'e_tradeseq_equivalents.csv', index=False)
display(TRADESEQ.round(3))

## 8 · (f) The taxonomy table and the overlap of species calls

Every row uses the same 1,513 pathways (fewer where an approach tests a subset) and the same three
partitions. Rows from earlier notebooks are read from their saved tables; nothing is refit.

In [ ]:
hits31 = pd.read_csv(nb31 / 'strategy_hits_all_partitions.csv')
tests37 = pd.read_csv(nb37 / 'tables' / 'matched_and_joint_tests_all_partitions.csv')
step45 = pd.read_csv(step45_file).set_index('pathway_id')
conv12 = pd.read_csv(nb12 / 'conventional_pt_ora.csv')
call_sets, rows = {}, []


def add(name, family, question, information, species, relabels, tested, source, external=''):
    call_sets[name] = set(species)
    ncdr, meets = rule(len(species), [len(r) for r in relabels], tested) if relabels else (np.nan, None)
    rows.append({'approach': name, 'family': family, 'question': question, 'information returned': information,
                 'tested': tested, 'species calls': len(species),
                 'relabeled calls': ', '.join(str(len(r)) for r in relabels) if relabels else 'not run',
                 'NCDR': ncdr, 'specific (rule)': meets, '∩ primary 22': len(set(species) & P22),
                 '∩ robust 36': len(set(species) & R36), 'external calibration': external, 'source': source})


def hits(strategy, partition):
    f = hits31[hits31.strategy.eq(strategy) & hits31.partition.eq(partition)]
    return set(f.pathway_id[f.hit.astype(bool)])


def joint(stat, partition):
    f = tests37[tests37.statistic.eq(stat) & tests37.partition.eq(partition)]
    return set(f.pathway_id[f.effect.gt(0) & f.q_joint.le(ALPHA)])


cross45 = pd.read_csv(nb45 / 'tables' / 'table_cross_species_draws_summary.csv').set_index('strategy')


def ext(strategy):
    r = cross45.loc[strategy]
    return f'cross-species draws: NCDR {r.ncdr_median:.2f}, {r.share_meeting_nb31_rule:.0%} meet rule'


add('Whole-PT pseudobulk DE + signed GSEA', 'discrete, average', 'average difference', 'yes/no, one sign',
    hits('1 · Whole-PT pseudobulk + GSEA', 'species'), [hits('1 · Whole-PT pseudobulk + GSEA', s) for s in swaps], 1508,
    'notebook 31', ext('O1 whole-PT pseudobulk + GSEA'))
add('S1/S2/S3 pseudobulk DE + signed GSEA', 'discrete, segment', 'difference within a segment', 'yes/no, sign per segment',
    hits('2 · S1/S2/S3 pseudobulk + GSEA', 'species'), [hits('2 · S1/S2/S3 pseudobulk + GSEA', s) for s in swaps], 1508,
    'notebook 31', ext('O2 S1/S2/S3 pseudobulk + GSEA'))
for variant, family, label in (('all genes', 'whole PT', 'Whole-PT pseudobulk DE + ORA'),
                               ('all genes', 'S1/S2/S3', 'S1/S2/S3 pseudobulk DE + ORA'),
                               ('equal probe counts', 'S1/S2/S3', 'S1/S2/S3 pseudobulk DE + ORA, equal-probe genes')):
    decision = ORA_DECISION.set_index('strategy').loc['ORA whole PT' if family == 'whole PT' else 'ORA S1/S2/S3']
    add(label, 'discrete, average' if family == 'whole PT' else 'discrete, segment',
        'over-representation among DE genes', 'yes/no, sign (per segment)', ora_sets[(variant, family, 'species')],
        [ora_sets[(variant, family, s)] for s in swaps],
        len(gene_sets) if variant == 'all genes' else len(restrict(gene_sets, equal_probe)), 'this notebook (a)',
        f"external: {decision.decision}; cross-species NCDR {decision['cross-species median NCDR']:.2f}")
segment_markers = set(conv12.pathway_id[conv12.scope.eq('segment_vs_rest') & conv12.q_family.le(ALPHA)])
add('Segment markers (segment vs rest) + ORA', 'discrete, zonation', 'what characterises each segment (species pooled)',
    'segment membership', segment_markers, [], 1513, 'notebook 12')
add('Species × S1/S2/S3 steps, coordinate-free (joint)', 'discrete, positional', 'does the segment profile differ',
    'yes/no; step pattern', set(step45.index[step45.called.astype(bool)]), [set(), set()], 1513, 'notebook 45',
    ext('P1 T_steps joint'))
add('Species × S1/S2/S3 steps on the spline (joint)', 'discrete, positional', 'does the segment profile differ',
    'yes/no; step pattern', joint('T_discrete_spatial', 'species'), [joint('T_discrete_spatial', s) for s in swaps], 1513, 'notebook 37')
add('Species × 6 coordinate bins (joint)', 'discretised coordinate', 'does the 6-bin profile differ', 'yes/no; 6-bin profile',
    joint('T_bins_spatial', 'species'), [joint('T_bins_spatial', s) for s in swaps], 1513, 'notebook 37')
add('T_spatial, matched rank-AUC', 'continuous, competitive', 'is the positional shape more species-dependent than matched genes',
    'yes/no; D(s); peak', hits('8 · Gene GAM × smooth position (T_spatial)', 'species'),
    [hits('8 · Gene GAM × smooth position (T_spatial)', s) for s in swaps], 1513, 'notebook 31')
add('T_spatial, joint test (primary)', 'continuous, competitive', 'as above, with inter-gene correlation',
    'yes/no; D(s); peak; shape class', joint('T_spatial', 'species'), [joint('T_spatial', s) for s in swaps], 1513, 'notebook 37')
add('T_spatial, preranked GSEA', 'continuous, competitive', 'as above, GSEA null', 'yes/no',
    hits('8c · T_spatial, preranked GSEA', 'species'), [hits('8c · T_spatial, preranked GSEA', s) for s in swaps], 1513, 'notebook 31')
add('T_spatial, size-only random sets', 'continuous, competitive', 'as above, unmatched null', 'yes/no',
    hits('8b · T_spatial, size-only random sets', 'species'), [hits('8b · T_spatial, size-only random sets', s) for s in swaps], 1513, 'notebook 31')
add('Beyond S1/S2/S3 steps (joint)', 'continuous, within-segment', 'does shape differ beyond label steps',
    'yes/no; within-segment component', joint('T_position_given_segments', 'species'),
    [joint('T_position_given_segments', s) for s in swaps], 1513, 'notebook 37 (notebook 57: identifiability)')
add('Gene GAM offset T_level (joint)', 'continuous, average', 'average difference', 'yes/no, sign',
    joint('T_level', 'species'), [joint('T_level', s) for s in swaps], 1513, 'notebook 37', ext('O3 T_level joint'))
add('Gene GAM total T_total (joint)', 'continuous, average', 'average or shape difference', 'yes/no',
    joint('T_total', 'species'), [joint('T_total', s) for s in swaps], 1513, 'notebook 37')
add('Pathway-score GAM, offset (structure level)', 'continuous, self-contained', 'does the activity score differ on average',
    'activity curve', hits('3a · Pathway score GAM, offset', 'species'), [hits('3a · Pathway score GAM, offset', s) for s in swaps], 1513, 'notebook 31')
add('Pathway-score GAM, shape (structure level)', 'continuous, self-contained', 'does the activity curve differ in shape',
    'activity curve', hits('3b · Pathway score GAM, shape', 'species'), [hits('3b · Pathway score GAM, shape', s) for s in swaps], 1513, 'notebook 31')
for score_name, label in (('mean_z', 'mean-z module score'), ('aucell', 'AUCell-like score')):
    add(f'Pathway activity ({label}), specimen × bin split plot', 'continuous, self-contained, specimen-level',
        'does the activity curve differ in shape, judged against specimens', 'activity curve per specimen',
        score_sets[score_name]['species'], [score_sets[score_name][s] for s in swaps], 1513, 'this notebook (b)')
split37 = {label: pd.read_csv(f) for label in partitions for f in
           glob.glob(str(nb37 / 'stage_cache' / f"split_matched_{label.replace(':', '_').replace('+', '_')}__*.csv"))[:1]}
if len(split37) == 3:
    add('Gene split plot (specimen × 8 bins), matched', 'continuous, specimen-level', 'positional difference against specimen × position',
        'yes/no', set(split37['species'].pathway_id[split37['species'].effect.gt(0) & split37['species'].q_empirical.le(ALPHA)]),
        [set(split37[s].pathway_id[split37[s].effect.gt(0) & split37[s].q_empirical.le(ALPHA)]) for s in swaps], 1513, 'notebook 37')
add(f'Curve modules (k = {K}) + ORA', 'continuous, shape families', 'which difference shapes recur and what fills them',
    'module shape (centroid) and member pathways', module_sets['species'], [module_sets[s] for s in swaps], 1513,
    f"this notebook (c); median LOSO ARI {MODULES['median LOSO ARI'].iloc[0]:.2f}")
for stat in ('concentration', 'location'):
    add(f'Divergence-peak {stat} enrichment', 'continuous, location', f'are member peaks {"concentrated" if stat == "concentration" else "displaced"}',
        'location of divergence', location_sets[stat]['species'], [location_sets[stat][s] for s in swaps],
        len(location['species']), 'this notebook (d)')
TAXONOMY = pd.DataFrame(rows)
TAXONOMY.to_csv(tables / 'f_taxonomy.csv', index=False)
display(TAXONOMY.drop(columns=['question', 'information returned']).round(2))

main = [name for name in call_sets if call_sets[name] and name not in ('Segment markers (segment vs rest) + ORA',)]
overlap = pd.DataFrame([[len(call_sets[a] & call_sets[b]) / max(len(call_sets[a] | call_sets[b]), 1) for b in main] for a in main],
                       index=main, columns=main)
overlap.to_csv(tables / 'f_jaccard_species_calls.csv')
membership_table = pd.DataFrame({name: [p in call_sets[name] for p in pathways] for name in call_sets}, index=pathways)
membership_table.insert(0, 'name', table37.name.reindex(pathways).to_numpy())
membership_table['primary 22'] = membership_table.index.isin(sorted(P22))
membership_table['robust 36'] = membership_table.index.isin(sorted(R36))
membership_table.to_csv(tables / 'f_species_calls_by_approach.csv')
print('Jaccard overlap of species calls (main approaches):')
display(overlap.round(2))

record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'k_modules': int(K),
          'ora_decision': ORA_DECISION.to_dict(orient='records'), 'scores': SCORES.to_dict(orient='records'),
          'modules': MODULES.to_dict(orient='records'), 'location': LOCATION.to_dict(orient='records')}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=float))

## 9 · Figure: the approaches side by side

Left: species calls against the mean of the two relabelings, per approach (log scale, so a count
of 0 is drawn at 0.8; the dashed line marks 5% of tested pathways). Right: which of the 36 robust principal-curve pathways each
approach calls (filled = called), with the 22 primary pathways marked.

In [ ]:
import matplotlib.pyplot as plt

shown = TAXONOMY[TAXONOMY['relabeled calls'].ne('not run')].copy()
shown['relabel mean'] = shown['relabeled calls'].str.split(', ').apply(lambda v: np.mean([float(x) for x in v]))
order = shown.sort_values(['family', 'species calls']).approach.tolist()
fig, (ax, bx) = plt.subplots(1, 2, figsize=(15, 9), gridspec_kw={'width_ratios': [1, 1.6]})
y = np.arange(len(order))
frame = shown.set_index('approach').loc[order]
ax.barh(y + .2, frame['species calls'].clip(lower=.8), height=.38, color='#2a78d6', label='species split')
ax.barh(y - .2, frame['relabel mean'].clip(lower=.8), height=.38, color='#eb6834', label='relabelings (mean)')
ax.axvline(.05 * 1513, color='#52514e', lw=1, ls='--')
ax.set_xscale('log')
ax.set_xlabel('Pathways called (log scale; ✓ = meets the specificity rule)', fontsize=8)
ax.legend(fontsize=8, frameon=False, loc='lower right')
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
robust_order = table37.loc[sorted(R36)].sort_values('peak_position').index
matrix = np.array([[p in call_sets[a] for a in order] for p in robust_order], float)
bx.imshow(matrix, aspect='auto', cmap=plt.matplotlib.colors.ListedColormap(['#ecebe7', '#2a78d6']), vmin=0, vmax=1)
bx.set_yticks(range(len(robust_order)), [('● ' if p in P22 else '   ') + str(table37.loc[p, 'name'])[:45] for p in robust_order], fontsize=6.5)
bx.set_xticks(range(len(order)), [str(i + 1) for i in range(len(order))], fontsize=7)
bx.set_xlabel('Approach (numbered as in the left panel, top = 1)', fontsize=8)
bx.set_title('Robust 36 (● primary 22), ordered by peak position', fontsize=9, loc='left')
ax.set_yticks(y, [f"{i + 1}. {a}{'  ✓' if s else ''}" for i, (a, s) in enumerate(zip(order, frame['specific (rule)']))], fontsize=7)
ax.invert_yaxis()
fig.tight_layout()
for ext_ in ('pdf', 'png'):
    fig.savefig(figures / f'approaches_side_by_side.{ext_}', dpi=200)
plt.show()